In [5]:
### 数据操作
import os
base_dir = './food11'
training_files = os.listdir(os.path.join(base_dir,'training'))
test_files = os.listdir(os.path.join(base_dir,'test'))
validation_files = os.listdir(os.path.join(base_dir,'validation'))
### 数据量统计
print(f"训练集合数据 {len(training_files)}")
print(f"测试集合数据 {len(test_files)}")
print(f"验证集合数据 {len(validation_files)}")

训练集合数据 9866
测试集合数据 3347
验证集合数据 3430


In [6]:
### 保证复现
import torch
import numpy as np
seed = 1000
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [11]:
### transformer
import torchvision.transforms as transforms
test_tfm = transforms.Compose([
    transforms.Resize([128,128]),
    transforms.ToTensor()
])

training_tfm = transforms.Compose([
    transforms.Resize([128,128]),
    transforms.ToTensor()
])

In [16]:
### 复写Dataset类
from torch.utils.data import Dataset
from PIL import Image
class FoodDataset(Dataset):
    def __init__(self, path, tfm = test_tfm, files = None):
        super().__init__() ### 父类中没有初始化方法
        if files is not None:
            self.files = files
        else:
            self.files = sorted([os.path.join(path,x) for x in os.listdir(path) if x.endswith('.jpg')])
        self.tfm = test_tfm
        print(f"{self.files[0]}")
    def __getitem__(self, idx):
        item = self.files[idx]
        fig = Image.open(item)
        fig = self.tfm(fig)
        try:
            label = int(item.split('/')[-1].split('_')[0])
        except:
            label = -1
        return fig,label
    def __len__(self):
        pass

In [17]:
train_data = FoodDataset(os.path.join(base_dir,'training'),training_tfm)
train_data[0]

./food11/training/0_0.jpg


(tensor([[[0.7412, 0.7373, 0.7294,  ..., 0.8353, 0.8353, 0.8314],
          [0.7412, 0.7373, 0.7255,  ..., 0.8471, 0.8471, 0.8431],
          [0.7333, 0.7333, 0.7333,  ..., 0.8510, 0.8510, 0.8471],
          ...,
          [0.2510, 0.2510, 0.2588,  ..., 0.8902, 0.9020, 0.8863],
          [0.2510, 0.2549, 0.2588,  ..., 0.9020, 0.9059, 0.9020],
          [0.2549, 0.2667, 0.2588,  ..., 0.9098, 0.9020, 0.9137]],
 
         [[0.8235, 0.8196, 0.8118,  ..., 0.8667, 0.8667, 0.8627],
          [0.8118, 0.8078, 0.7961,  ..., 0.8667, 0.8667, 0.8667],
          [0.7961, 0.7922, 0.7961,  ..., 0.8784, 0.8784, 0.8706],
          ...,
          [0.2588, 0.2588, 0.2667,  ..., 0.9490, 0.9569, 0.9451],
          [0.2588, 0.2627, 0.2667,  ..., 0.9608, 0.9647, 0.9608],
          [0.2627, 0.2745, 0.2667,  ..., 0.9686, 0.9608, 0.9725]],
 
         [[0.9882, 0.9843, 0.9804,  ..., 0.9725, 0.9686, 0.9686],
          [0.9725, 0.9686, 0.9569,  ..., 0.9804, 0.9804, 0.9765],
          [0.9373, 0.9333, 0.9333,  ...,